|<h2>Substack post:</h2>|<h1><a href="https://mikexcohen.substack.com/" target="_blank">Exploring the Wav2Vec2 model</a></h1>|
|-|:-:|
|<h2>Teacher:<h2>|<h1>Mike X Cohen, <a href="https://sincxpress.com" target="_blank">sincxpress.com</a></h1>|

<br>

<i>Using the code without reading the post may lead to confusion or errors.</i>

In [ ]:
## References:
# Model on Hugging Face: https://huggingface.co/facebook/wav2vec2-base-960h
# Model paper: https://arxiv.org/abs/2006.13979

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

from IPython.display import Audio

In [ ]:
### matplotlib adjustments for dark mode

# svg plots (higher-res)
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

plt.rcParams.update({
    'figure.facecolor': '#282a2c',#'#191919',#
    'figure.edgecolor': '#282a2c',
    'axes.facecolor':   '#282a2c',
    'axes.edgecolor':   '#DDE2F4',
    'axes.labelcolor':  '#DDE2F4',
    'xtick.color':      '#DDE2F4',
    'ytick.color':      '#DDE2F4',
    'text.color':       '#DDE2F4',
    'axes.spines.right': False,
    'axes.spines.top':   False,
    'axes.titleweight': 'bold',
    'axes.labelweight': 'bold',
    'savefig.dpi':300
})

# **Import and inspect the model and tokenizer**

In [ ]:
modelID = 'facebook/wav2vec2-base-960h'

processor = Wav2Vec2Processor.from_pretrained(modelID)
model = Wav2Vec2ForCTC.from_pretrained(modelID)
model.eval()

In [ ]:
# the tokenizer
processor.tokenizer

In [ ]:
# decode all tokens
for t in range(processor.tokenizer.vocab_size):
  print(f'Token #{t:2} is "{processor.decode(t)}"')

# **Exploring the feature extractor (convolution stack)**

In [ ]:
for m in model.wav2vec2.feature_extractor.conv_layers:
  print(m)

In [ ]:
# convenience variable
num_conv = len(model.wav2vec2.feature_extractor.conv_layers)

# example of how to access the weights
model.wav2vec2.feature_extractor.conv_layers[1].conv.weight.shape

In [ ]:
# FYI about the torch convolution object
C = torch.nn.Conv1d(1,512,kernel_size=10)

# for a 10-point filter, the result of convolution with a 10-point signal is just one point
C(torch.ones(1,10)).shape

In [ ]:
# which filters to visualize
filts = np.linspace(0,model.config.conv_dim[0]-1,7,dtype=int)


plt.figure(figsize=(10,4))

for i in range(len(filts)):

  # extract and normalize this filter
  y = model.wav2vec2.feature_extractor.conv_layers[0].conv.weight[filts[i],0,:].detach()
  y = (y-y.min()) / (y.max()-y.min())

  # and visualize with a y-axis offset
  plt.plot(range(-9,1),i+y,linewidth=2)


plt.gca().set(xlabel='Time point (indices)',title='A few filter kernels',
              ylabel='Filter coefficients (a.u.)',yticks=[])
plt.show()

In [ ]:
# images of the full kernel matrices
_,axs = plt.subplots(1,3,figsize=(12,4))
for i in range(3):
  axs[i].imshow(model.wav2vec2.feature_extractor.conv_layers[1].conv.weight[:,:,i].detach(),
           vmin=-.1,vmax=.1,aspect='auto',cmap='plasma')
  axs[i].set(title=f'Filter coefficient {i}')

plt.show()

# **Temporal downsampling and aggregation**

In [ ]:
# required sampling rate
srate = processor.feature_extractor.sampling_rate

# initialize vectors
kernlen = np.ones(num_conv+1)
stride = np.ones(num_conv+1)

# loop over layers
for layeri in range(num_conv):

  # get parameters (kernel size and stride)
  k = model.wav2vec2.feature_extractor.conv_layers[layeri].conv.kernel_size[0]
  s = model.wav2vec2.feature_extractor.conv_layers[layeri].conv.stride[0]

  # calculate integration window
  kernlen[layeri+1] = kernlen[layeri] + (k-1)*stride[layeri]
  stride[layeri+1] = stride[layeri]*s


# and plot
plt.figure(figsize=(10,4))
plt.plot(kernlen*1000/srate,'wo-',markersize=10,markerfacecolor=[.7,.9,.7],label='Kernel len')
plt.plot(stride*1000/srate,'ws-',markersize=10,markerfacecolor=[.9,.7,.7],label='Stride')

plt.legend()
plt.gca().set(xlabel='Convolution layer',ylabel='Time (ms)',xticks=range(num_conv+1),
              xticklabels=['Input'] + [f'Layer {i}' for i in range(num_conv)])
plt.show()

# **Import data (speech audio file) and run through model**

In [ ]:
# download the dataset
dataset = torchaudio.datasets.LIBRISPEECH(root='.',url='test-clean',download=True)

# pick one sample and confirm its sampling rate
waveform,srate,orig_text, *_ = dataset[0]
timevec = np.arange(0,len(waveform[0]))/srate
print(f'Sampling rate is {srate} Hz (needs to be 16 kHz)\n')

print(orig_text)
Audio(waveform,rate=srate)

In [ ]:
# define a hook function to store convolution activations
activations = {}

def implant_hook(modnum):
  def hook(module,input,output):
    activations[f'convlayer_{modnum}'] = output.detach().numpy()
  return hook


# implant hooks
handles = []
modnum = 0
for module in model.wav2vec2.feature_extractor.conv_layers:
  h = module.register_forward_hook(implant_hook(modnum))
  modnum += 1
  handles.append(h)

In [ ]:
# run the audio through the model
outs = model(waveform)
outs.keys(), activations.keys()

In [ ]:
# check layer sizes
print(f'      Input has shape {list(waveform.shape)}')
for k,v in activations.items():
  print(f'{k} has shape {list(v.shape)}')

In [ ]:
# visualize the activations
fig,axs = plt.subplots(2,4,figsize=(12,4))
axs = axs.flatten()

for ai in range(num_conv):

  # get the activations for this layer (first sequence in the batch)
  X = activations[f'convlayer_{ai}'][0]

  # draw the heatmap with color limits
  cmin,cmax = np.percentile(X,[5,95])
  h = axs[ai].imshow(X,aspect='auto',cmap='magma',vmin=cmin,vmax=cmax)
  fig.colorbar(h,ax=axs[ai],pad=.02,fraction=.04)
  axs[ai].set(xlabel='Time (index)',ylabel='Embeddings',title=f'Layer {ai}')

# disable unused axis
axs[-1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# pick one time point for later embeddings visualization
oneTimeIdx = [np.argmin(abs(timevec-.75))] # .75 seconds, converted to index from original time vector

plt.figure(figsize=(12,4))

# scale and plot the input signal
y = waveform[0]
y = (y-y.min()) / (y.max()-y.min())
plt.plot(timevec,y+1+num_conv,label=f'In ($N$ = {len(y)})')


for li in range(num_conv):

  # extract the scaled L1-mean of the convolution layer
  y = abs(activations[f'convlayer_{li}'][0]).mean(axis=0)
  y = (y-y.min()) / (y.max()-y.min())

  # create an interpolated time vector
  timey = np.linspace(timevec[0],timevec[-1],len(y))
  oneTimeIdx.append(np.argmin(abs(timey-.75))) # store one time point for later visualization

  # and plot
  plt.plot(timey,y+num_conv-li,label=f'L{li} ($N$ = {len(y)})')
  plt.axvline(timey[oneTimeIdx[-1]],color='w',linewidth=.1)


plt.gca().set(xlim=timevec[[0,-1]],yticks=[],xlabel='Time (sec.)',ylabel='Amplitude (a.u.)')
plt.legend(bbox_to_anchor=(1,1))

plt.tight_layout()
plt.show()

In [ ]:
# all convolution dimensions for one time point
plt.figure(figsize=(12,4))

# get the default color sequence to match the previous figure
dftcolors = plt.rcParams['axes.prop_cycle'].by_key()['color']


for li in range(num_conv):

  # extract the scaled average of the convolution layer
  y = abs(activations[f'convlayer_{li}'][0,:,oneTimeIdx[li+1]])
  y = (y-y.min()) / (y.max()-y.min())

  # plot the embeddings
  plt.plot(y+num_conv-li,'s-',markersize=3,linewidth=.5,color=dftcolors[li+1],label=f'L{li}')

plt.gca().set(xlim=[-1,model.config.conv_dim[0]],yticks=[],xlabel='Embedding dim.',ylabel='Amplitude (a.u.)')
plt.legend(bbox_to_anchor=(1,1))
plt.show()

In [ ]:
# distributions
fig,axs = plt.subplots(1,2,figsize=(12,4))

for ai in range(num_conv):

  # histogram of raw activations
  y,x = np.histogram(activations[f'convlayer_{ai}'].flatten(),bins=20,density=True)
  axs[0].plot(x[:-1],y,label=f'Layer {ai}',linewidth=2,color=dftcolors[ai+1])

  # histogram of scaled activations
  X = activations[f'convlayer_{ai}'].flatten()
  X = (X-X.min()) / (X.max()-X.min())
  y,x = np.histogram(X,bins=np.linspace(0,1.001,20),density=True)
  axs[1].plot(x[:-1],y,label=f'Layer {ai}',linewidth=2,color=dftcolors[ai+1])

for i,a in enumerate(axs):
  a.legend()
  a.set(ylabel='Probability density',yscale='log',xlabel=f'Activation ({["raw","min-max scaled"][i]})')

plt.tight_layout()
plt.show()

# **Decoding the max-tokens**

In [ ]:
# remove hook handles to avoid overwriting
for h in handles: h.remove()

In [ ]:
# using a different (shorter) stimulus
waveform,srate,orig_text,*_ = dataset[30]
print(orig_text)

In [ ]:
# forward pass with hidden states
outs = model(waveform,output_hidden_states=True)

In [ ]:
# decode the model's output
predicted_tokens = outs.logits.argmax(dim=-1)
processor.batch_decode(predicted_tokens)

In [ ]:
outs.logits.shape

In [ ]:
maxlogits = outs.logits.max(dim=-1)
maxlogits

In [ ]:
# "raw" decoding of each time step

for t in maxlogits.indices[0]:
  print(f'Token #{t:2} is "{processor.decode(t)}"')

In [ ]:
processor.tokenizer.decode(maxlogits.indices[0])

# **Logit lens analysis of hidden states**

In [ ]:
# need the lm_head layer for decoding
print(f'Final logits size: {list(outs.logits.shape)}')
print(f'Hidden state size: {list(outs.hidden_states[0].shape)}')
print(f' lm_head(hs) size: {list(model.lm_head(outs.hidden_states[0]).shape)}')

In [ ]:
# "logit lens" is the decoded output of each hidden layer (transformer block)
plt.figure(figsize=(10,3))

for hi in range(len(outs.hidden_states)):

  # convert hidden states to token predictions
  logits = model.lm_head(outs.hidden_states[hi]).detach()

  # get maximum predicted letter indices
  predicted_tokens = logits.argmax(dim=-1)

  # get softmax probability for max predictions
  softmaxs = logits.softmax(dim=-1)[0].max(dim=-1)[0].mean()
  plt.plot(hi,softmaxs,'wh',markersize=12,markerfacecolor=plt.cm.plasma(softmaxs))

  # decode into text
  text = processor.batch_decode(predicted_tokens)

  # print the results
  print(f'Hidden layer {hi:2} ({softmaxs.mean():.1%}): {text[0]}')


plt.gca().set(xlabel='Transformer layer',ylabel='Average softmax probability')
plt.show()